In [1]:
import psutil
import os

def print_resource_usage(stage_name: str, ram_limit_gb=1.0, cpu_limit_m=500):
    """
    Выводит текущее потребление RAM и CPU и предупреждает о превышении лимитов.
    """
    p = psutil.Process(os.getpid())
    
    # --- Память ---
    # memory_info().rss возвращает Resident Set Size - актуально занимаемую физическую память
    ram_gb = p.memory_info().rss / (1024 ** 3)
    
    # --- CPU ---
    # .cpu_percent() показывает загрузку с момента последнего вызова.
    # interval=0.1 дает небольшую задержку для более точного измерения.
    cpu_percent = p.cpu_percent(interval=0.1)
    cpu_m = cpu_percent * 10 # 100% CPU = 1 core = 1000m. 1% = 10m.
    
    print(f"--- {stage_name} ---")
    print(f"    RAM: {ram_gb:.3f} GB")
    print(f"    CPU: {cpu_m:.1f}m")
    
    # --- Предупреждения ---
    if ram_gb > ram_limit_gb:
        print(f"    !!! ВНИМАНИЕ: Потребление RAM ({ram_gb:.3f} GB) превысило лимит в {ram_limit_gb} GB.")
    if cpu_m > cpu_limit_m:
         print(f"    !!! ВНИМАНИЕ: Потребление CPU ({cpu_m:.1f}m) превысило лимит в {cpu_limit_m}m.")
    print("-" * (len(stage_name) + 6))

In [2]:
import pandas as pd
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    TrainerCallback # Импортируем базовый класс для коллбэка
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import numpy as np
import joblib

# --- Кастомный коллбэк для мониторинга ---
class ResourceMonitorCallback(TrainerCallback):
    """Коллбэк, который выводит потребление ресурсов в конце каждой эпохи."""
    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = state.epoch
        print_resource_usage(f"Конец эпохи {epoch:.0f}")

#  Подготовка данных (Модель №1 "Специалист") ---
print("--- ОБУЧЕНИЕ МОДЕЛИ №1 ('Специалист') ---")
print_resource_usage("Старт скрипта")

df_specific = pd.read_csv('../data/TRAINING_DATASET_SPECIFIC.csv')
print_resource_usage("Данные загружены в Pandas")

X = df_specific['processed_text'].fillna('')
y = df_specific['label']

label_encoder_specific = LabelEncoder()
y_encoded = label_encoder_specific.fit_transform(y)
joblib.dump(label_encoder_specific, 'label_encoder_specific.joblib')

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Загрузка модели и токенизатора ---
MODEL_NAME = 'cointegrated/rubert-tiny'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Мониторинг ресурсы ДО и ПОСЛЕ загрузки модели, чтобы увидеть скачок RAM
print_resource_usage("Перед загрузкой модели")
model_specific = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_encoder_specific.classes_)
)
print_resource_usage("ПОСЛЕ загрузки модели") # <-- Это самый важный замер!

# --- 3. Создание датасетов PyTorch ---
# (Класс TextDataset оставляем из предыдущего кода)
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels): self.encodings, self.labels = encodings, labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    def __len__(self): return len(self.labels)

# Trainer от Hugging Face имеет встроенные прогресс-бары, tqdm не нужен
train_encodings = tokenizer(X_train.tolist(), truncation=True, padding=True, max_length=512)
test_encodings = tokenizer(X_test.tolist(), truncation=True, padding=True, max_length=512)
train_dataset = TextDataset(train_encodings, y_train.tolist())
test_dataset = TextDataset(test_encodings, y_test.tolist())

# Обучение ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {'f1': f1_score(labels, predictions, average='macro')}

training_args = TrainingArguments(
    output_dir='./results_specific',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    evaluation_strategy="epoch",  # Оценивать качество в конце каждой эпохи
    save_strategy="epoch",        # Сохранять модель в конце каждой эпохи
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50, # <-- Оставим логирование частым, чтобы видеть прогресс
    # флаг, чтобы не забивать диск старыми чекпоинтами
    save_total_limit=2 # только 2 (лучший и самый свежий)
)

trainer = Trainer(
    model=model_specific,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[ResourceMonitorCallback()] # <-- Коллбэк сюда!
)

trainer.train()

# --- Сохранение ---
SAVE_PATH_SPECIFIC = './saved_model_specific_best'
trainer.save_model(SAVE_PATH_SPECIFIC)
tokenizer.save_pretrained(SAVE_PATH_SPECIFIC)
print_resource_usage("Обучение Модели №1 завершено")

--- ОБУЧЕНИЕ МОДЕЛИ №1 ('Специалист') ---
--- Старт скрипта ---
    RAM: 0.350 GB
    CPU: 0.0m
-------------------
--- Данные загружены в Pandas ---
    RAM: 0.367 GB
    CPU: 0.0m
-------------------------------
--- Перед загрузкой модели ---
    RAM: 0.383 GB
    CPU: 0.0m
----------------------------


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


--- ПОСЛЕ загрузки модели ---
    RAM: 0.429 GB
    CPU: 5017.0m
    !!! ВНИМАНИЕ: Потребление CPU (5017.0m) превысило лимит в 500m.
---------------------------


D:\projects\context\contextual_search\venv\Lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,F1
1,0.016100,0.013807,0.999239
2,0.005300,0.005418,0.999239
3,0.003500,0.004196,0.999239


--- Конец эпохи 1 ---
    RAM: 0.951 GB
    CPU: 8644.0m
    !!! ВНИМАНИЕ: Потребление CPU (8644.0m) превысило лимит в 500m.
-------------------
--- Конец эпохи 2 ---
    RAM: 0.962 GB
    CPU: 9807.0m
    !!! ВНИМАНИЕ: Потребление CPU (9807.0m) превысило лимит в 500m.
-------------------
--- Конец эпохи 3 ---
    RAM: 0.965 GB
    CPU: 8976.0m
    !!! ВНИМАНИЕ: Потребление CPU (8976.0m) превысило лимит в 500m.
-------------------
--- Обучение Модели №1 завершено ---
    RAM: 0.966 GB
    CPU: 0.0m
----------------------------------


In [3]:
import pandas as pd
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    TrainerCallback # Импортируем базовый класс для коллбэка
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import numpy as np
import joblib

# --- Кастомный коллбэк для мониторинга ---
class ResourceMonitorCallback(TrainerCallback):
    """Коллбэк, который выводит потребление ресурсов в конце каждой эпохи."""
    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = state.epoch
        print_resource_usage(f"Конец эпохи {epoch:.0f}")

#  Подготовка данных (МОДЕЛЬ №2 "Сортировщик") ---
print("--- ОБУЧЕНИЕ МОДЕЛИ №2 'Сортировщик' ---")
print_resource_usage("Старт скрипта")

df_specific = pd.read_csv('../data/TRAINING_DATASET_GENERAL.csv')
print_resource_usage("Данные загружены в Pandas")

X = df_specific['processed_text'].fillna('')
y = df_specific['label']

label_encoder_specific = LabelEncoder()
y_encoded = label_encoder_specific.fit_transform(y)
joblib.dump(label_encoder_specific, 'label_encoder_general.joblib')

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Загрузка модели и токенизатора ---
MODEL_NAME = 'cointegrated/rubert-tiny'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Мониторинг ресурсы ДО и ПОСЛЕ загрузки модели, чтобы увидеть скачок RAM
print_resource_usage("Перед загрузкой модели")
model_specific = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_encoder_specific.classes_)
)
print_resource_usage("ПОСЛЕ загрузки модели") # <-- Это самый важный замер!

# --- 3. Создание датасетов PyTorch ---
# (Класс TextDataset оставляем из предыдущего кода)
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels): self.encodings, self.labels = encodings, labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    def __len__(self): return len(self.labels)

# Trainer от Hugging Face имеет встроенные прогресс-бары, tqdm не нужен
train_encodings = tokenizer(X_train.tolist(), truncation=True, padding=True, max_length=512)
test_encodings = tokenizer(X_test.tolist(), truncation=True, padding=True, max_length=512)
train_dataset = TextDataset(train_encodings, y_train.tolist())
test_dataset = TextDataset(test_encodings, y_test.tolist())

# Обучение ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {'f1': f1_score(labels, predictions, average='macro')}

training_args = TrainingArguments(
    output_dir='./results_general',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    evaluation_strategy="epoch",  # Оценивать качество в конце каждой эпохи
    save_strategy="epoch",        # Сохранять модель в конце каждой эпохи
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50, # <-- Оставим логирование частым, чтобы видеть прогресс
    # флаг, чтобы не забивать диск старыми чекпоинтами
    save_total_limit=2 # только 2 (лучший и самый свежий)
)

trainer = Trainer(
    model=model_specific,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[ResourceMonitorCallback()] # <-- Коллбэк сюда!
)

trainer.train()

# --- Сохранение ---
SAVE_PATH_SPECIFIC = './saved_model_general_best'
trainer.save_model(SAVE_PATH_SPECIFIC)
tokenizer.save_pretrained(SAVE_PATH_SPECIFIC)
print_resource_usage("Обучение Модели №2 завершено")

--- ОБУЧЕНИЕ МОДЕЛИ №2 'Сортировщик' ---
--- Старт скрипта ---
    RAM: 0.966 GB
    CPU: 0.0m
-------------------
--- Данные загружены в Pandas ---
    RAM: 0.968 GB
    CPU: 0.0m
-------------------------------
--- Перед загрузкой модели ---
    RAM: 0.970 GB
    CPU: 0.0m
----------------------------


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


--- ПОСЛЕ загрузки модели ---
    RAM: 0.970 GB
    CPU: 4300.0m
    !!! ВНИМАНИЕ: Потребление CPU (4300.0m) превысило лимит в 500m.
---------------------------


D:\projects\context\contextual_search\venv\Lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,F1
1,0.385000,0.435718,0.809104
2,0.365100,0.370690,0.840536
3,0.354200,0.370214,0.847975


--- Конец эпохи 1 ---
    RAM: 1.167 GB
    CPU: 7955.0m
    !!! ВНИМАНИЕ: Потребление RAM (1.167 GB) превысило лимит в 1.0 GB.
    !!! ВНИМАНИЕ: Потребление CPU (7955.0m) превысило лимит в 500m.
-------------------
--- Конец эпохи 2 ---
    RAM: 1.176 GB
    CPU: 7884.0m
    !!! ВНИМАНИЕ: Потребление RAM (1.176 GB) превысило лимит в 1.0 GB.
    !!! ВНИМАНИЕ: Потребление CPU (7884.0m) превысило лимит в 500m.
-------------------
--- Конец эпохи 3 ---
    RAM: 1.178 GB
    CPU: 8145.0m
    !!! ВНИМАНИЕ: Потребление RAM (1.178 GB) превысило лимит в 1.0 GB.
    !!! ВНИМАНИЕ: Потребление CPU (8145.0m) превысило лимит в 500m.
-------------------
--- Обучение Модели №2 завершено ---
    RAM: 1.111 GB
    CPU: 0.0m
    !!! ВНИМАНИЕ: Потребление RAM (1.111 GB) превысило лимит в 1.0 GB.
----------------------------------


In [4]:
import torch
import joblib
from transformers import AutoTokenizer, AutoModelForSequenceClassification


# --- Загрузка всех компонентов ---
print_resource_usage("Старт пайплайна")

TOKENIZER_PATH = './saved_model_specific_best'
MODEL_SPECIFIC_PATH = './saved_model_specific_best'
ENCODER_SPECIFIC_PATH = 'label_encoder_specific.joblib'
MODEL_GENERAL_PATH = './saved_model_general_best'
ENCODER_GENERAL_PATH = 'label_encoder_general.joblib'

device = torch.device("cpu") # Для прода с CPU-лимитом лучше тестировать на CPU

# Загружаем всё в память и делаем замеры
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
print_resource_usage("Загружен токенизатор")

model_specific = AutoModelForSequenceClassification.from_pretrained(MODEL_SPECIFIC_PATH).to(device)
print_resource_usage("Загружена Модель №1")

model_general = AutoModelForSequenceClassification.from_pretrained(MODEL_GENERAL_PATH).to(device)
print_resource_usage("Загружена Модель №2 (ОБЕ МОДЕЛИ В ПАМЯТИ)") # <-- ФИНАЛЬНЫЙ ЗАМЕР!

encoder_specific = joblib.load(ENCODER_SPECIFIC_PATH)
encoder_general = joblib.load(ENCODER_GENERAL_PATH)

print("\nВсе компоненты иерархического классификатора загружены.")

--- Старт пайплайна ---
    RAM: 1.111 GB
    CPU: 0.0m
    !!! ВНИМАНИЕ: Потребление RAM (1.111 GB) превысило лимит в 1.0 GB.
---------------------
--- Загружен токенизатор ---
    RAM: 1.111 GB
    CPU: 0.0m
    !!! ВНИМАНИЕ: Потребление RAM (1.111 GB) превысило лимит в 1.0 GB.
--------------------------
--- Загружена Модель №1 ---
    RAM: 1.141 GB
    CPU: 5040.0m
    !!! ВНИМАНИЕ: Потребление RAM (1.141 GB) превысило лимит в 1.0 GB.
    !!! ВНИМАНИЕ: Потребление CPU (5040.0m) превысило лимит в 500m.
-------------------------
--- Загружена Модель №2 (ОБЕ МОДЕЛИ В ПАМЯТИ) ---
    RAM: 1.168 GB
    CPU: 4872.0m
    !!! ВНИМАНИЕ: Потребление RAM (1.168 GB) превысило лимит в 1.0 GB.
    !!! ВНИМАНИЕ: Потребление CPU (4872.0m) превысило лимит в 500m.
-----------------------------------------------

Все компоненты иерархического классификатора загружены.
